# Front-end data for "The 2026 mayoral election: neighbourhood types"

This notebook pre-processes the data behind the graphics on the page `src/routes/the-2026-mayoral-election-neighbourhood-types`, so the page only has to draw and never has to compute. **Each section builds the file for one graphic and writes it as JSON or GeoJSON into that route's `data/` folder.** Sections are added as the graphics are defined.

| Section | Graphic | Output file |
| --- | --- | --- |
| 1 | Ranked lists of the strongest neighbourhood correlates of Chow's and Bradford's 2026 vote share | `correlations_top.json` |
| 2 | Chow's vote and the Chinese share of residents, 2014, 2023 and 2026, in Old Toronto and the suburbs | `chinese_correlation.json` |
| 3 | Chow's and Bradford's 2026 votes and the Jewish and Muslim shares of residents | `religion_correlation.json` |
| 4 | Where Chow's added votes came from: tracts she held, flipped or did not win (map with inset bars) | `new_votes_groups.json`, `new_votes_tracts.geojson`, `new_votes_municipalities.geojson` |
| 5 | Who lives in the neighbourhoods where Chow's vote was highest: strip plots for the top fifth of tracts | `chow_top_fifth_strips.json` |

Everything describes **census tracts**, not individual voters, and the 2026 vote is the **election-day vote** apportioned to 2021 census tracts (advance and mail-in ballots have no tract). Results, apportionment and the correlation script come from Zack Taylor's `data/tor_elections_2023_2026` workflow (methods in its `METHODS.md`).

In [1]:
import json
import warnings
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from shapely.geometry.polygon import orient
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
REPO_ROOT = Path.cwd().resolve().parents[1]
ELECTIONS_DIR = REPO_ROOT / "data" / "tor_elections_2023_2026"
CORRELATES_CSV = ELECTIONS_DIR / "output" / "plots" / "correlates_2026.csv"           # written by plot_correlates_2026.R: every variable, both candidates
RESULTS_DTA = ELECTIONS_DIR / "output" / "resultsbyctuid2021_2026.dta"                # election-day results by 2021 tract (pct{candidate}2026)
SOCGEO_DIR = REPO_ROOT / "src" / "routes" / "social-geography-and-the-mayoral-vote" / "data"
CENSUS_PARQUET = SOCGEO_DIR / "census.parquet"   # 2021 census by tract, used only to re-check the CSV
ROUTE_DATA = REPO_ROOT / "src" / "routes" / "the-2026-mayoral-election-neighbourhood-types" / "data"                 # every output below is written here

# Section 1: strongest correlates of each candidate's 2026 vote share
CANDIDATES = {"chow": "Chow", "bradford": "Bradford"}                                 # key -> name used in the CSV
RESULT_COLUMNS = {"chow": "pctchow_olivia2026", "bradford": "pctbradford_brad2026"}
TOP_N = 10                                                                            # strongest positive correlations kept per candidate
EXCLUDE_VARIABLES = []        # variables to leave out of the lists; e.g. ["walkscore", "transitscore", "bikescore"] for the proprietary scores
N_TRACTS = 585

# Values read off the chart this section reproduces (correlation rounded to 2 decimals; labels in the CSV's variable codes)
IMAGE = {
    "chow": {"marsmaclcomm": .62, "jwmdbike": .59, "occ_arsp": .59, "rlgnnone": .55, "jwmdwalk": .52, "walkscore": .52, "transitscore": .46, "marsnmclnevr": .45,
             "hhtnrent": .39, "pop_dens": .39, "agec3064": .37, "jwmdtran": .36, "bikescore": .35, "rlgnothr": .35, "dwtpapt0": .33, "vminvisijapn": .32},
    "bradford": {"jwmdauto": .61, "marsmaclmarr": .61, "dwtpsdet": .58, "ihat_med": .44, "hhtnown_": .42, "occ_heal": .41, "ihat_avg": .41, "hcov_avg": .39,
                 "rlgnjewi": .38, "dist": .36, "ethmeurosoutital": .32},
}
IMAGE_TOLERANCE = .006                                                                # rounding in the chart plus the third decimal

# Section 2: Chow's vote share and the Chinese visible-minority share of tract residents
CHINESE_BY_AREA_CSV = ELECTIONS_DIR / "output" / "plots" / "chow_chinese_correlation_by_area.csv"   # plot_chow_chinese_correlation.R: by former City of Toronto / suburbs
CHINESE_CITYWIDE_CSV = ELECTIONS_DIR / "output" / "plots" / "chow_chinese_correlation.csv"          # same script: all tracts, three census years
TRACTS_GEOJSON = SOCGEO_DIR / "tracts.geojson"                                                      # 2021 tracts with the `oldcity` field the script's area split uses
CHINESE_ELECTIONS = ["2014", "2023", "2026"]
CHINESE_AREAS = {"City of Toronto": "Old Toronto", "Suburbs": "Suburbs"}                            # script's name -> label on the chart
CHINESE_CENSUS = "2021"                                                                            # the script uses the 2021 census for all three elections
CHINESE_TOLERANCE = .005                                                                           # tolerance for the re-computation

# Section 3: Chow's and Bradford's 2026 vote share and the Jewish and Muslim shares of tract residents
RELIGION_CSV = ELECTIONS_DIR / "output" / "plots" / "religion_correlations_2026.csv"                 # plot_religion_correlations_2026.R
RELIGION_VARS = {"Jewish": "rlgnjewi_pct", "Muslim": "rlgnmusl_pct"}                                 # 2021 census religion shares of residents
RELIGION_RESULTS = {"chow": ("pctchow_olivia2026", "Chow"), "bradford": ("pctbradford_brad2026", "Bradford")}   # role -> (2026 results column, name in the CSV)
RELIGION_TOLERANCE = .005                                                                           # tolerance for the re-computation against the script's CSV
HIGH_SHARE = .20                                                                                    # "concentrated" tracts: more than 20% of residents

# Section 4: where Chow's votes grew, 2023 to 2026
RESULTS_DTA_2023 = ELECTIONS_DIR / "output" / "resultsbyctuid2021_2023.dta"           # 2023 by-election, election-day and special-ballot votes by 2021 tract
MAYOR_GPKG = REPO_ROOT / "data" / "election_results" / "2026-results-clean" / "mayor_2023_2026_ct.gpkg"   # the earlier tract layer: used only to re-check the votes
HEAD_TO_HEAD = {2023: ("a_chow_olivia2023", "a_bailao_ana2023"), 2026: ("a_chow_olivia2026", "a_bradford_brad2026")}   # Chow's and her main opponent's votes, by tract
GAIN_GROUPS = {"held": "Chow both times", "flipped": "Flipped to Chow", "trailing": "Opponent both times", "other": "Other"}   # key -> label on the page
MAP_EPSG = 32617                  # UTM 17N, metres: simplification only
MAP_TOLERANCE_M = 25              # boundary simplification for the map; neighbouring tracts are simplified separately, so hairline gaps are possible (hidden by the map's thin outlines)
MAP_DECIMALS = 4                  # degrees of longitude and latitude kept (about 8 m)
MUNI_GEOJSON = REPO_ROOT / "src" / "routes" / "the-2026-mayoral-election-local-landslides" / "data" / "former_municipalities.geo.json"   # the series' six former municipalities, with a label point each
# Totals reported in the earlier notebook (02_chow_coalition_expansion, Section 1), used to confirm this rebuild: tracts, Chow's pooled 2023 and 2026 share (%), added votes
EXPECTED_GROUPS = {"held": (256, 45.8, 65.7, 108917), "flipped": (146, 30.6, 53.3, 57525), "trailing": (177, 21.7, 39.3, 47869)}
EXPECTED_OTHER = (6, 1394)        # reverse transitions (2) and exact ties (4): tracts, added votes

# Section 5: the top fifth of tracts by Chow's 2026 vote share, against all tracts, on five neighbourhood measures
CURATED_CSV = REPO_ROOT / "analysis" / "toronto_election_turnout" / "data" / "archive" / "modelling" / "processed" / "toronto_ct_modelling_curated.csv"   # 2021 census median age (dem_median_age)
FIVE_TORONTOS_VALUES = REPO_ROOT / "src" / "routes" / "five-torontos" / "data" / "ct_values.json"                        # used only to cross-check the shares
TOP_FRACTION = 5                  # the top fifth: the 585 // 5 = 117 tracts with the highest Chow share in 2026
# key, label, source column, factor to the plotted unit, plotted domain, tick step, unit. A tract outside the domain is not drawn (the medians still use every tract)
STRIP_VARS = [
    {"key": "renters", "label": "% renters", "column": "hhtnrent_pct", "factor": 100, "domain": [0, 100], "tick": 20, "unit": "percent"},
    {"key": "income", "label": "Median household income", "column": "ihat_med__nc", "factor": 1, "domain": [40000, 160000], "tick": 20000, "unit": "dollars"},
    {"key": "age", "label": "Median age", "column": "dem_median_age", "factor": 1, "domain": [30, 60], "tick": 5, "unit": "years"},
    {"key": "car", "label": "% commuting by car", "column": "jwmdauto_pct", "factor": 100, "domain": [0, 100], "tick": 20, "unit": "percent"},
    {"key": "vismin", "label": "% visible minority", "column": "vminvisi_pct", "factor": 100, "domain": [0, 100], "tick": 20, "unit": "percent"},
]

# Re-check: CSV variable -> column of the repo's 2021 census file (shares and levels). Walk/Transit/Bike Scores, average dwelling value, Average rent and
# the education rebuild are not in this file, and `agec3064` is defined differently there, so they cannot be re-computed here.
CENSUS_MAP = {
    "marsmaclcomm": "marsmaclcomm_pct", "marsmaclmarr": "marsmaclmarr_pct", "marsnmclnevr": "marsnmclnevr_pct", "jwmdbike": "jwmdbike_pct", "jwmdwalk": "jwmdwalk_pct",
    "jwmdtran": "jwmdtran_pct", "jwmdauto": "jwmdauto_pct", "occ_arsp": "occ_arts_pct", "occ_heal": "occ_heal_pct", "rlgnnone": "rlgnnone_pct", "rlgnothr": "rlgnothr_pct",
    "rlgnjewi": "rlgnjewi_pct", "rlgnchricath_tot": "rlgnchricath_tot_pct", "hhtnrent": "hhtnrent_pct", "hhtnown_": "hhtnown__pct", "dwtpsdet": "dwtpsdet_pct",
    "pop_dens": "popdsqkm__nc", "dist": "dist__nc", "ihat_med": "ihat_med__nc", "ihat_avg": "ihat_avg__nc", "vminvisijapn": "vminvisijapn_pct",
    "ethmeurosoutital": "ethmeurosoutital_pct",
}
RECHECK_TOLERANCE = .006                                                              # distance from City Hall differs by up to .005 between the two files

## 1. Strongest correlates of each candidate's 2026 vote share

**Output:** `correlations_top.json`, the data for the two ranked lists.

**Source.** `correlates_2026.csv`, written by Zack Taylor's `plot_correlates_2026.R` (`data/tor_elections_2023_2026`, documented in `METHODS.md`, "Tract correlates of 2026 vote share"). That script has one row per variable and candidate, so this notebook reads its result rather than recomputing it, since two of its inputs are not in this repository (the harmonized census file with Walk, Transit and Bike Scores, and the detailed 2021 education tables).

**Method (from the script).**
- Pearson correlation across the **585 2021 census tracts**, **unweighted**, between a tract characteristic and each candidate's **election-day vote share** (`pct{candidate}2026` in `resultsbyctuid2021_2026.dta`). Shares are shares of valid votes in the tract.
- Variables: the 2021 rows of the harmonized census file, as shares (averages, medians and population density as they stand), plus distance from City Hall and Walk, Transit and Bike Scores. This is the curated harmonized list, not every census code.
- University degree and high school or less are rebuilt from the detailed 2021 tables (the harmonized file's 2021 values are not shares of one population); both stay below .30.
- The original chart showed only positive correlations of .30 or more; here the **ten strongest positive correlations** per candidate are kept, which is the same ordering.

Three cells follow: the CSV is loaded and the lists built; the lists are checked against the earlier chart; and 22 of the correlations are **recomputed** from the repo's own results and census files as an independent check.

In [3]:
corr = pd.read_csv(CORRELATES_CSV)
assert set(corr.candidate) == set(CANDIDATES.values()) and (corr.groupby("candidate").size() == corr.groupby("candidate").size().iloc[0]).all()
POOL = int(corr.groupby("candidate").size().iloc[0])
assert corr.n.max() == N_TRACTS
TOP = {}
for key, name in CANDIDATES.items():
    c = corr[(corr.candidate == name) & ~corr.variable.isin(EXCLUDE_VARIABLES)].sort_values("r", ascending=False)
    TOP[key] = c.head(TOP_N).reset_index(drop=True)
    print(f"\n{name}: {POOL} variables in the CSV; top {TOP_N} positive correlations (excluded: {EXCLUDE_VARIABLES or 'none'})")
    print(TOP[key].assign(r=TOP[key].r.round(3))[["variable", "label", "r", "n"]].to_string())
    print("  next five:", [(l, round(r, 3)) for l, r in zip(c.label.iloc[TOP_N:TOP_N + 5], c.r.iloc[TOP_N:TOP_N + 5])])


Chow: 88 variables in the CSV; top 10 positive correlations (excluded: none)
       variable                                label      r    n
0  marsmaclcomm           Marital status: Common-law  0.619  585
1      jwmdbike                     Commute: Bicycle  0.593  583
2      occ_arsp  Occupation: Arts, recreation, sport  0.593  583
3      rlgnnone                          No religion  0.550  583
4      jwmdwalk                        Commute: Walk  0.517  583
5     walkscore                           Walk Score  0.515  585
6  transitscore                        Transit Score  0.455  585
7  marsnmclnevr        Marital status: Never married  0.453  585
8      hhtnrent                     Rented dwellings  0.391  583
9      pop_dens                   Population density  0.389  585
  next five: [('Population age 30–64', 0.369), ('Commute: Transit', 0.359), ('Bike Score', 0.35), ('Ethnic origin: British', 0.349), ('Religion: Other', 0.345)]

Bradford: 88 variables in the CSV; top 10 pos

In [4]:
# Check the earlier chart's rows against the CSV (the chart rounded to 2 decimals)
rows = []
for key, vals in IMAGE.items():
    c = corr[corr.candidate == CANDIDATES[key]].set_index("variable")
    for var, ref in vals.items():
        rows.append({"candidate": key, "variable": var, "label": c.loc[var, "label"], "chart": ref, "csv": round(c.loc[var, "r"], 3), "difference": round(c.loc[var, "r"] - ref, 3)})
CHECK = pd.DataFrame(rows)
assert CHECK.difference.abs().max() <= IMAGE_TOLERANCE, CHECK[CHECK.difference.abs() > IMAGE_TOLERANCE]
print(f"All {len(CHECK)} rows of the earlier chart agree with the CSV within {IMAGE_TOLERANCE}; largest difference {CHECK.difference.abs().max():.3f}.")
for key, name in CANDIDATES.items():
    c = corr[(corr.candidate == name) & (corr.r >= .30)]
    new = c[~c.variable.isin(IMAGE[key])].sort_values("r", ascending=False)
    print(f"{name}: correlations of .30 or more in the CSV but not on the earlier chart:", [(l, round(r, 3)) for l, r in zip(new.label, new.r)])

All 27 rows of the earlier chart agree with the CSV within 0.006; largest difference 0.005.
Chow: correlations of .30 or more in the CSV but not on the earlier chart: [('Ethnic origin: British', 0.349)]
Bradford: correlations of .30 or more in the CSV but not on the earlier chart: [('Religion: Catholic', 0.363)]


In [5]:
# Independent re-computation of the CSV's correlations from files that are in the repository
res = pd.read_stata(RESULTS_DTA).assign(ct=lambda d: d.ctuid2021.astype(str)).set_index("ct")
cen = pd.read_parquet(CENSUS_PARQUET).query("year == 2021").set_index("ct")
assert len(res) == N_TRACTS and set(res.index) == set(cen.index)
rows = []
for key, name in CANDIDATES.items():
    c = corr[corr.candidate == name].set_index("variable")
    for var, col in tqdm(CENSUS_MAP.items(), desc=f"re-check {name}", leave=False):
        d = pd.DataFrame({"x": cen[col], "y": res[RESULT_COLUMNS[key]]}).dropna()
        rows.append({"candidate": key, "variable": var, "csv": round(c.loc[var, "r"], 3), "recomputed": round(d.x.corr(d.y), 3), "n": len(d)})
RE = pd.DataFrame(rows).assign(difference=lambda d: (d.recomputed - d.csv).round(3))
assert RE.difference.abs().max() <= RECHECK_TOLERANCE, RE[RE.difference.abs() > RECHECK_TOLERANCE]
print(f"{len(CENSUS_MAP)} variables x 2 candidates recomputed from resultsbyctuid2021_2026.dta and the 2021 census file: largest difference from the CSV {RE.difference.abs().max():.3f}.")
print("Not recomputable here (inputs not in the repository):", sorted(set(corr.variable) - set(CENSUS_MAP))[:12], "...")

22 variables x 2 candidates recomputed from resultsbyctuid2021_2026.dta and the 2021 census file: largest difference from the CSV 0.005.
Not recomputable here (inputs not in the repository): ['agec0019', 'agec065p', 'agec2029', 'agec3064', 'bikescore', 'dwpc00001960', 'dwpc19611990', 'dwpc19912021', 'dwptothr', 'dwtpapt0', 'dwtpapt5', 'edu_highschool'] ...


In [6]:
ROUTE_DATA.mkdir(parents=True, exist_ok=True)
out = {"meta": {"year": 2026, "tracts": N_TRACTS, "pool": POOL, "top_n": TOP_N,
                "measure": "Pearson r between a tract characteristic and the candidate's share of valid election-day votes, unweighted, across 2021 census tracts",
                "source": "data/tor_elections_2023_2026/output/plots/correlates_2026.csv (plot_correlates_2026.R)",
                "built_by": "analysis/election_2026/03_neighbourhood_types_page_data.ipynb"}}
for key, name in CANDIDATES.items():
    out[key] = {"name": name, "rows": [{"id": r.variable, "label": r.label, "r": round(float(r.r), 3), "n": int(r.n)} for r in TOP[key].itertuples()]}
path = ROUTE_DATA / "correlations_top.json"
path.write_text(json.dumps(out, indent=1, ensure_ascii=False) + "\n")
back = json.loads(path.read_text())
assert all(len(back[k]["rows"]) == TOP_N for k in CANDIDATES) and all(a["r"] >= b["r"] for k in CANDIDATES for a, b in zip(back[k]["rows"], back[k]["rows"][1:]))
print(f"wrote {path.relative_to(REPO_ROOT)} ({path.stat().st_size:,} bytes)")

wrote src/routes/the-2026-mayoral-election-neighbourhood-types/data/correlations_top.json (2,518 bytes)


## 2. Chow's vote and the Chinese share of residents

**Output:** `chinese_correlation.json`, the data for the dot-and-interval chart.

**Source.** `chow_chinese_correlation_by_area.csv` (and `chow_chinese_correlation.csv` for the all-tract figures), written by Zack Taylor's `plot_chow_chinese_correlation.R` (`METHODS.md`, "Chow's vote share and Chinese visible minority share").

**Method (from the script).**
- Pearson correlation across 2021 census tracts, **unweighted**, between Chow's election-day vote share (`pctchow_olivia{year}` in `resultsbyctuid2021_{year}.dta`) and the share of residents who are **Chinese visible minority** (`vminvisichin_pct`).
- The **2021 census is used for all three elections** (the CSV also holds 2011 and 2016 census versions, which run up to .05 higher).
- 95% confidence intervals are from `cor.test()` (Fisher's *z*). They treat tracts as independent, and neighbouring tracts are alike, so they are probably too narrow.
- **Old Toronto** is the former City of Toronto (the `oldcity` field); the **suburbs** are every other tract (East York, Etobicoke, North York, Scarborough and York). Tracts without Chinese-share data are dropped, so the areas have 178 and 405 tracts.

The next cells load the script's results, **recompute** every correlation and interval from files in the repository, and write the JSON.

In [7]:
area = pd.read_csv(CHINESE_BY_AREA_CSV).astype({"election": str})
city = pd.read_csv(CHINESE_CITYWIDE_CSV).astype({"election": str, "census": str}).query("census == @CHINESE_CENSUS")
assert set(area.election) == set(CHINESE_ELECTIONS) and set(area.area) == set(CHINESE_AREAS) and len(area) == 6 and len(city) == 3
display(area.round(3))

,election,area,n,mean_chinese,r,low,high
0,2014,City of Toronto,178,0.09,0.202,0.056,0.339
1,2014,Suburbs,405,0.12,0.193,0.097,0.285
2,2023,City of Toronto,178,0.09,0.237,0.093,0.371
3,2023,Suburbs,405,0.12,0.427,0.344,0.503
4,2026,City of Toronto,178,0.09,0.233,0.089,0.368
5,2026,Suburbs,405,0.12,0.116,0.019,0.211


In [8]:
# Recompute r and its Fisher-z 95% interval from the repository's own results, census and tract files
cen = pd.read_parquet(CENSUS_PARQUET).query("year == 2021").set_index("ct")["vminvisichin_pct"]
tr = gpd.read_file(TRACTS_GEOJSON)[["ct", "oldcity"]].set_index("ct")
print("`oldcity` values:", tr.oldcity.value_counts().to_dict())
tr["area"] = np.where(tr.oldcity == "Toronto", "City of Toronto", "Suburbs")
Z = 1.959964                                                                          # qnorm(.975), as in R's cor.test

def r_ci(x, y):
    r = np.corrcoef(x, y)[0, 1]; se = 1 / np.sqrt(len(x) - 3)
    return r, np.tanh(np.arctanh(r) - Z * se), np.tanh(np.arctanh(r) + Z * se)

rows = []
for election in tqdm(CHINESE_ELECTIONS, desc="elections"):
    res = pd.read_stata(ELECTIONS_DIR / "output" / f"resultsbyctuid2021_{election}.dta").assign(ct=lambda d: d.ctuid2021.astype(str)).set_index("ct")[f"pctchow_olivia{election}"]
    d = pd.DataFrame({"chow": res, "chinese": cen, "area": tr.area}).dropna()
    for name in ["All tracts"] + list(CHINESE_AREAS):
        s = d if name == "All tracts" else d[d.area == name]
        r, lo, hi = r_ci(s.chow, s.chinese); rows.append({"election": election, "area": name, "n": len(s), "r": r, "low": lo, "high": hi})
RE = pd.DataFrame(rows)
ref = pd.concat([area[["election", "area", "n", "r", "low", "high"]], city.assign(area="All tracts")[["election", "area", "n", "r", "low", "high"]]])
CMP = RE.merge(ref, on=["election", "area"], suffixes=("_here", "_script"))
for c in ("r", "low", "high"): CMP[f"d_{c}"] = (CMP[f"{c}_here"] - CMP[f"{c}_script"]).round(4)
display(CMP[["election", "area", "n_here", "n_script", "r_here", "r_script", "d_r", "d_low", "d_high"]].round(3))
assert (CMP.n_here == CMP.n_script).all(), "tract counts differ from the script's"
assert CMP[["d_r", "d_low", "d_high"]].abs().max().max() <= CHINESE_TOLERANCE, "recomputed values differ from the script's by more than the tolerance"
print(f"Recomputed all {len(CMP)} correlations and intervals from repository files: largest difference {CMP[['d_r', 'd_low', 'd_high']].abs().max().max():.4f}.")

`oldcity` values: {'Toronto': 179, 'North York': 142, 'Scarborough': 126, 'Etobicoke': 79, 'York': 32, 'East York': 27}


,election,area,n_here,n_script,r_here,r_script,d_r,d_low,d_high
0,2014,All tracts,583,583,0.038,0.038,-0.0,-0.0,0.0
1,2014,City of Toronto,178,178,0.202,0.202,0.0,-0.0,0.0
2,2014,Suburbs,405,405,0.193,0.193,0.0,-0.0,0.0
3,2023,All tracts,583,583,0.229,0.229,0.0,-0.0,0.0
4,2023,City of Toronto,178,178,0.237,0.237,0.0,0.0,0.0
5,2023,Suburbs,405,405,0.427,0.427,-0.0,-0.0,0.0
6,2026,All tracts,583,583,0.044,0.044,-0.0,-0.0,0.0
7,2026,City of Toronto,178,178,0.233,0.233,0.0,0.0,0.0
8,2026,Suburbs,405,405,0.116,0.116,-0.0,-0.0,0.0


Recomputed all 9 correlations and intervals from repository files: largest difference 0.0000.


In [9]:
out = {"meta": {"census_year": int(CHINESE_CENSUS), "level": 0.95,
                "measure": "Pearson r across 2021 census tracts, unweighted, between Chow's share of valid election-day votes and the Chinese visible-minority share of residents (2021 census for all elections); 95% interval from Fisher's z",
                "areas": {k: v for k, v in CHINESE_AREAS.items()},
                "source": "data/tor_elections_2023_2026/output/plots/chow_chinese_correlation_by_area.csv (plot_chow_chinese_correlation.R)",
                "built_by": "analysis/election_2026/03_neighbourhood_types_page_data.ipynb"},
       "rows": [{"election": int(r.election), "area": CHINESE_AREAS[r.area], "n": int(r.n), "r": round(float(r.r), 3), "low": round(float(r.low), 3), "high": round(float(r.high), 3)} for r in area.itertuples()],
       "citywide": [{"election": int(r.election), "n": int(r.n), "r": round(float(r.r), 3), "low": round(float(r.low), 3), "high": round(float(r.high), 3)} for r in city.itertuples()]}
path = ROUTE_DATA / "chinese_correlation.json"
path.write_text(json.dumps(out, indent=1, ensure_ascii=False) + "\n")
back = json.loads(path.read_text())
assert len(back["rows"]) == 6 and all(r["low"] < r["r"] < r["high"] for r in back["rows"]) and len(back["citywide"]) == 3
print(f"wrote {path.relative_to(REPO_ROOT)} ({path.stat().st_size:,} bytes)")

wrote src/routes/the-2026-mayoral-election-neighbourhood-types/data/chinese_correlation.json (1,576 bytes)


## 3. Chow's and Bradford's 2026 vote and the Jewish and Muslim shares of residents

**Output:** `religion_correlation.json`, the data for the second dot-and-interval chart.

**Source and method.** `religion_correlations_2026.csv`, written by Zack Taylor's `plot_religion_correlations_2026.R` (`METHODS.md`, "2026 vote share and Jewish and Muslim population share"): Pearson correlation across 2021 census tracts, unweighted, between each candidate's 2026 election-day vote share (`pct{candidate}2026` in `resultsbyctuid2021_2026.dta`) and the **2021 census** Jewish or Muslim share of residents (`rlgnjewi_pct`, `rlgnmusl_pct`), with a 95% interval from `cor.test()`.

The next cells recompute the four correlations and intervals from files in the repository (Section 2's `r_ci()` gives the Pearson *r* and Fisher-*z* interval), check them against the script's CSV, and write the JSON from the script's values.

In [10]:
# r_ci() is defined in the Section 2 re-check cell above (Pearson r and its Fisher-z interval, as in R's cor.test)
script = pd.read_csv(RELIGION_CSV)
assert len(script) == 4 and set(script.religion) == set(RELIGION_VARS) and set(script.candidate) == {n for _, n in RELIGION_RESULTS.values()}
census_rel = pd.read_parquet(CENSUS_PARQUET).query("year == 2021").set_index("ct")[list(RELIGION_VARS.values())]
res = pd.read_stata(ELECTIONS_DIR / "output" / "resultsbyctuid2021_2026.dta").assign(ct=lambda d: d.ctuid2021.astype(str)).set_index("ct")
rows = []
for religion, col in tqdm(RELIGION_VARS.items(), desc="religions"):
    for role, (rescol, name) in RELIGION_RESULTS.items():
        d = pd.DataFrame({"x": census_rel[col], "y": res[rescol]}).dropna()
        r, lo, hi = r_ci(d.x, d.y)
        s = script[(script.religion == religion) & (script.candidate == name)].iloc[0]
        rows.append({"religion": religion, "candidate": name, "n": len(d), "n_script": int(s.n), "r": r, "r_script": s.r, "low": lo, "low_script": s.low, "high": hi, "high_script": s.high})
REL = pd.DataFrame(rows)
display(REL.round(3))
diff = max((REL[c] - REL[f"{c}_script"]).abs().max() for c in ("r", "low", "high"))
assert (REL.n == REL.n_script).all() and diff <= RELIGION_TOLERANCE, diff
print(f"The four recomputed correlations and intervals match the script's CSV (largest difference {diff:.4f}).")
share = census_rel.dropna()
SHARES = {rel: {"median_pct": round(float(share[col].median() * 100), 1), "tracts_over_20pct": int((share[col] > HIGH_SHARE).sum())} for rel, col in RELIGION_VARS.items()}
print("Concentration of each group across tracts:", SHARES)

,religion,candidate,n,n_script,r,r_script,low,low_script,high,high_script
0,Jewish,Chow,583,583,-0.316,-0.316,-0.387,-0.387,-0.241,-0.241
1,Jewish,Bradford,583,583,0.383,0.383,0.311,0.311,0.450,0.450
2,Muslim,Chow,583,583,0.067,0.067,-0.014,-0.014,0.148,0.148
3,Muslim,Bradford,583,583,-0.157,-0.157,-0.235,-0.235,-0.077,-0.077


The four recomputed correlations and intervals match the script's CSV (largest difference 0.0000).
Concentration of each group across tracts: {'Jewish': {'median_pct': 1.0, 'tracts_over_20pct': 24}, 'Muslim': {'median_pct': 6.5, 'tracts_over_20pct': 40}}


In [11]:
out = {"meta": {"year": 2026, "census_year": 2021, "level": 0.95, "tracts": int(script.n.max()),
                "measure": "Pearson r across 2021 census tracts, unweighted, between a candidate's share of valid 2026 election-day votes and the Jewish or Muslim share of residents (2021 census); 95% interval from Fisher's z",
                "source": "data/tor_elections_2023_2026/output/plots/religion_correlations_2026.csv (plot_religion_correlations_2026.R), re-computed here",
                "built_by": "analysis/election_2026/03_neighbourhood_types_page_data.ipynb"},
       "rows": [{"religion": r.religion, "role": role, "candidate": r.candidate, "n": int(r.n), "r": round(float(r.r), 3), "low": round(float(r.low), 3), "high": round(float(r.high), 3)}
                for r in script.itertuples() for role, (_, name) in RELIGION_RESULTS.items() if name == r.candidate],
       "shares": SHARES}
path = ROUTE_DATA / "religion_correlation.json"
path.write_text(json.dumps(out, indent=1, ensure_ascii=False) + "\n")
back = json.loads(path.read_text())
assert len(back["rows"]) == 4 and all(r["low"] < r["r"] < r["high"] for r in back["rows"])
print(f"wrote {path.relative_to(REPO_ROOT)} ({path.stat().st_size:,} bytes)")

wrote src/routes/the-2026-mayoral-election-neighbourhood-types/data/religion_correlation.json (1,269 bytes)


## 4. Where Chow's added votes came from, 2023 to 2026

**Outputs:** `new_votes_groups.json` (the inset bars), `new_votes_tracts.geojson` (the map) and `new_votes_municipalities.geojson` (the former municipalities' outlines and label points, copied from the local-landslides page's file).

**Method.** Each census tract is sorted by who led the **head-to-head contest** in each election: Chow against Bailão in the 2023 by-election and Chow against Bradford in 2026 (her votes minus the main opponent's, from the tract results). *Chow both times* and *Opponent both times* are tracts the same side led in both; *Flipped to Chow* are tracts her opponent led in 2023 and she led in 2026; *Other* collects the six tracts that do not fit (two where she lost a 2023 lead and four exact ties). For each group the chart uses her **pooled** share of all valid votes in 2023 and 2026 (summed votes over summed valid votes, so large tracts count more) and her **added votes** (2026 minus 2023, summed over the group's tracts).

The votes are the tract estimates for **mapped poll votes** (about 78% of valid votes in 2023 and 76% in 2026, no advance or mail-in ballots), so the 2023 baseline is *not* the official 37.2% to 32.5% result. The groups and totals reproduce the earlier analysis notebook (`02_chow_coalition_expansion`, Section 1) and are re-checked against it below.

In [12]:
def head_to_head(path, year):
    d = pd.read_stata(path)
    chow, opp = HEAD_TO_HEAD[year]
    cand = [c for c in d.columns if c.startswith("a_") and c.endswith(str(year))]      # one column per candidate: votes apportioned to the tract
    return pd.DataFrame({"ct": d.ctuid2021.astype(str), f"valid{year}": d[cand].sum(axis=1), f"chow{year}": d[chow], f"opp{year}": d[opp]}).set_index("ct")

G4 = head_to_head(RESULTS_DTA_2023, 2023).join(head_to_head(RESULTS_DTA, 2026), how="inner")
assert len(G4) == N_TRACTS and G4.notna().all().all()

# independent check of the votes against the earlier tract layer
mayor = gpd.read_file(MAYOR_GPKG, layer="mayor_ct").assign(ct=lambda d: d.ctuid2021.astype(str)).set_index("ct")
for y, slug in ((2023, "bailao_ana"), (2026, "bradford_brad")):
    for mine, theirs in ((f"valid{y}", f"mayor{y}_votes_candidates"), (f"chow{y}", f"mayor{y}_chow_olivia_votes"), (f"opp{y}", f"mayor{y}_{slug}_votes")):
        assert (G4[mine] - mayor.loc[G4.index, theirs]).abs().max() < 1e-6, (mine, theirs)
print("Votes by tract match the earlier tract layer (valid, Chow and main opponent, both elections).")

lead23, lead26 = np.sign((G4.chow2023 - G4.opp2023).round(9)), np.sign((G4.chow2026 - G4.opp2026).round(9))
G4["group"] = np.select([(lead23 > 0) & (lead26 > 0), (lead23 < 0) & (lead26 > 0), (lead23 < 0) & (lead26 < 0)], ["held", "flipped", "trailing"], "other")

def pooled4(d):
    return {"tracts": len(d), "chow23": d.chow2023.sum() / d.valid2023.sum() * 100, "chow26": d.chow2026.sum() / d.valid2026.sum() * 100, "added": d.chow2026.sum() - d.chow2023.sum(),
            "valid23": d.valid2023.sum(), "valid26": d.valid2026.sum()}
GAIN = pd.DataFrame({k: pooled4(G4[G4.group == k]) for k in GAIN_GROUPS}).T
ALL4 = pooled4(G4)
GAIN["share_of_increase"] = GAIN.added / ALL4["added"] * 100
GAIN["gain_pp"] = GAIN.chow26 - GAIN.chow23
display(GAIN.round(1)); print("All 585 tracts:", {k: round(float(v), 1) for k, v in ALL4.items()})

assert int(GAIN.tracts.sum()) == N_TRACTS and np.isclose(GAIN.added.sum(), ALL4["added"]) and np.isclose(GAIN.share_of_increase.sum(), 100)
for k, (n, a, b, add) in EXPECTED_GROUPS.items():
    r = GAIN.loc[k]
    assert int(r.tracts) == n and abs(r.chow23 - a) < .06 and abs(r.chow26 - b) < .06 and abs(r.added - add) < 1, (k, r.to_dict())
assert (int(GAIN.loc["other", "tracts"]), round(GAIN.loc["other", "added"])) == EXPECTED_OTHER
print("Group sizes, pooled shares and added votes match the earlier analysis notebook.")

Votes by tract match the earlier tract layer (valid, Chow and main opponent, both elections).


,tracts,chow23,chow26,added,valid23,valid26,share_of_increase,gain_pp
held,256.0,45.8,65.7,108917.0,243296.0,335147.0,50.5,20.0
flipped,146.0,30.6,53.3,57525.0,143297.0,190013.0,26.7,22.8
trailing,177.0,21.7,39.3,47869.0,176052.0,219461.0,22.2,17.5
other,6.0,32.3,48.4,1394.0,3753.0,5387.0,0.6,16.1


All 585 tracts: {'tracts': 585.0, 'chow23': 34.4, 'chow26': 54.7, 'added': 215705.0, 'valid23': 566398.0, 'valid26': 750008.0}
Group sizes, pooled shares and added votes match the earlier analysis notebook.


In [13]:
GROUP_OUT = [{"key": k, "label": GAIN_GROUPS[k], "tracts": int(GAIN.loc[k, "tracts"]), "chow23": round(float(GAIN.loc[k, "chow23"]), 1), "chow26": round(float(GAIN.loc[k, "chow26"]), 1),
              "added_votes": int(round(GAIN.loc[k, "added"])), "share_of_increase": round(float(GAIN.loc[k, "share_of_increase"]), 1)} for k in GAIN_GROUPS]
out = {"meta": {"measure": "Chow's pooled share of valid mapped votes (summed votes over summed valid votes) and her added votes, by who led the Chow vs. main-opponent contest in each election",
                "elections": "2023 by-election (Chow vs. Bailao) and 2026 (Chow vs. Bradford)", "tracts": N_TRACTS,
                "citywide": {"chow23": round(float(ALL4["chow23"]), 1), "chow26": round(float(ALL4["chow26"]), 1), "added_votes": int(round(ALL4["added"])), "valid_growth_pct": round(float((ALL4["valid26"] / ALL4["valid23"] - 1) * 100), 1)},
                "source": "data/tor_elections_2023_2026/output/resultsbyctuid2021_{2023,2026}.dta, re-checked against data/election_results/2026-results-clean/mayor_2023_2026_ct.gpkg",
                "built_by": "analysis/election_2026/03_neighbourhood_types_page_data.ipynb"},
       "groups": GROUP_OUT}
path = ROUTE_DATA / "new_votes_groups.json"
path.write_text(json.dumps(out, indent=1, ensure_ascii=False) + "\n")
back = json.loads(path.read_text())
assert sum(g["tracts"] for g in back["groups"]) == N_TRACTS and abs(sum(g["share_of_increase"] for g in back["groups"]) - 100) < .15
print(f"wrote {path.relative_to(REPO_ROOT)} ({path.stat().st_size:,} bytes)")
print(json.dumps(back["meta"]["citywide"]))

wrote src/routes/the-2026-mayoral-election-neighbourhood-types/data/new_votes_groups.json (1,347 bytes)
{"chow23": 34.4, "chow26": 54.7, "added_votes": 215705, "valid_growth_pct": 32.4}


In [14]:
tracts = gpd.read_file(TRACTS_GEOJSON)[["ct", "geometry"]].set_index("ct")
assert set(tracts.index) == set(G4.index)
proj = tracts.to_crs(MAP_EPSG)
proj["geometry"] = proj.geometry.simplify(MAP_TOLERANCE_M, preserve_topology=True)
simple = proj.to_crs(4326).join(G4[["group"]]).reset_index().rename(columns={"group": "g"})[["ct", "g", "geometry"]]
assert simple.geometry.is_valid.all() and not simple.geometry.is_empty.any() and len(simple) == N_TRACTS
def clockwise(g):
    # d3-geo reads a counter-clockwise outer ring as "everything except this polygon", so the outer rings are written clockwise
    return orient(g, sign=-1.0) if g.geom_type == "Polygon" else g.__class__([orient(p, sign=-1.0) for p in g.geoms])
simple["geometry"] = simple.geometry.apply(clockwise)
path = ROUTE_DATA / "new_votes_tracts.geojson"
path.unlink(missing_ok=True)
simple.to_file(path, driver="GeoJSON", COORDINATE_PRECISION=MAP_DECIMALS, RFC7946="NO")     # RFC 7946 would rewind the rings counter-clockwise again
back = gpd.read_file(path)
raw = json.loads(path.read_text())
def first_ring(geom): return geom["coordinates"][0] if geom["type"] == "Polygon" else geom["coordinates"][0][0]
def signed_area(ring): return sum((x1 * y2 - x2 * y1) for (x1, y1), (x2, y2) in zip(ring, ring[1:] + ring[:1])) / 2
assert all(signed_area(first_ring(f["geometry"])) < 0 for f in raw["features"]), "outer rings must run clockwise for d3-geo"
assert len(back) == N_TRACTS and back.g.value_counts().to_dict() == G4.group.value_counts().to_dict()
area_change = abs(back.to_crs(MAP_EPSG).area.sum() / tracts.to_crs(MAP_EPSG).area.sum() - 1)
print(f"wrote {path.relative_to(REPO_ROOT)} ({path.stat().st_size:,} bytes); total area changed by {area_change:.3%} after simplifying")

wrote src/routes/the-2026-mayoral-election-neighbourhood-types/data/new_votes_tracts.geojson (214,678 bytes); total area changed by 0.057% after simplifying


In [15]:
# the outlines and label points of the six former municipalities, as the local-landslides page's change map draws them; the copy keeps this page self-contained
muni = gpd.read_file(MUNI_GEOJSON)
assert len(muni) == 6 and {"name", "label_lon", "label_lat"} <= set(muni.columns) and muni.crs.to_epsg() == 4326
muni["geometry"] = muni.geometry.apply(clockwise)
path = ROUTE_DATA / "new_votes_municipalities.geojson"
path.unlink(missing_ok=True)
muni[["name", "label_lon", "label_lat", "geometry"]].to_file(path, driver="GeoJSON", COORDINATE_PRECISION=MAP_DECIMALS + 1, RFC7946="NO")
raw = json.loads(path.read_text())
assert len(raw["features"]) == 6 and all(signed_area(first_ring(f["geometry"])) < 0 for f in raw["features"])
print(f"wrote {path.relative_to(REPO_ROOT)} ({path.stat().st_size:,} bytes): {', '.join(muni.name)}")

wrote src/routes/the-2026-mayoral-election-neighbourhood-types/data/new_votes_municipalities.geojson (41,585 bytes): York, North York, East York, Scarborough, Etobicoke, Old Toronto


## 5. The neighbourhoods where Chow's vote was highest

**Output:** `chow_top_fifth_strips.json`, the data for the strip plots (one row per measure, a dot per census tract).

**Method.** Tracts are ranked by Chow's share of valid 2026 votes (Section 4's mapped poll votes) and the **top fifth, 117 of 585 tracts**, are highlighted. Each strip shows every tract's value of one 2021 census measure (the share of households that rent, median household income, median age, the share of commuters who drive, the visible-minority share of residents), with dotted lines at the median of **all tracts** and of the **top fifth**. The medians are unweighted medians of tract values and use every tract with data, including any beyond the plotted range (income above $160,000 and ages outside 30 to 60 are not drawn). They describe neighbourhoods, not individual voters.

In [16]:
share26 = (G4.chow2026 / G4.valid2026 * 100).rename("chow_share")
cens = pd.read_parquet(CENSUS_PARQUET).query("year == 2021").set_index("ct")
age = pd.read_csv(CURATED_CSV, dtype={"ct_id": str}).set_index("ct_id")["dem_median_age"]
S5 = pd.DataFrame({"chow_share": share26})
for v in STRIP_VARS:
    src = age if v["column"] == "dem_median_age" else cens[v["column"]]
    S5[v["key"]] = src.reindex(S5.index) * v["factor"]
assert len(S5) == N_TRACTS and S5.chow_share.notna().all(), "tracts must line up across the files"
n_top = N_TRACTS // TOP_FRACTION
S5["top"] = S5.chow_share.rank(ascending=False, method="first") <= n_top                    # ties broken by order; the cut is 117 tracts exactly
cut = S5.loc[S5.top, "chow_share"].min()
assert int(S5.top.sum()) == n_top and S5.loc[~S5.top, "chow_share"].max() <= cut
print(f"Top fifth: {n_top} tracts with Chow's 2026 share of {cut:.1f}% or more (citywide tract median {S5.chow_share.median():.1f}%).")

# cross-check the three shares against the five-torontos page's tract file (the same 2021 census; the medians should agree closely)
ft = pd.DataFrame(json.load(open(FIVE_TORONTOS_VALUES))).set_index("ct_id")
for key, other in (("renters", "pct_renter"), ("car", "pct_commute_car"), ("vismin", "pct_visible_minority")):
    both = pd.concat([S5[key], ft[other]], axis=1).dropna()
    gap = (both.iloc[:, 0] - both.iloc[:, 1]).abs()
    print(f"{key}: {len(both)} tracts in common, median absolute gap {gap.median():.2f} points, correlation {both.corr().iloc[0, 1]:.3f}")
    assert both.corr().iloc[0, 1] > .99 and gap.median() < 1.5, key

rows = []
for v in STRIP_VARS:
    x = S5[v["key"]]; lo, hi = v["domain"]
    drawn = x.between(lo, hi)
    rows.append({"measure": v["label"], "tracts with data": int(x.notna().sum()), "outside range (not drawn)": int((x.notna() & ~drawn).sum()),
                 "median, all tracts": x.median(), "median, top fifth": x[S5.top].median()})
display(pd.DataFrame(rows).set_index("measure").round(1))

Top fifth: 117 tracts with Chow's 2026 share of 66.8% or more (citywide tract median 53.5%).
renters: 583 tracts in common, median absolute gap 0.03 points, correlation 1.000
car: 582 tracts in common, median absolute gap 0.04 points, correlation 0.999
vismin: 583 tracts in common, median absolute gap 0.04 points, correlation 1.000


,tracts with data,outside range (not drawn),"median, all tracts","median, top fifth"
measure,,,,
% renters,583,0,43.0,53.0
Median household income,583,24,86000.0,84000.0
Median age,585,9,41.2,38.4
% commuting by car,583,0,63.2,44.1
% visible minority,583,0,53.2,36.1


In [17]:
def clean(x, v):
    return [None if pd.isna(a) else (int(round(a)) if v["unit"] == "dollars" else round(float(a), 1)) for a in x]

out = {"meta": {"measure": "Unweighted medians of 2021 census tract values: all 585 tracts, and the 117 tracts with the highest Chow share of valid 2026 mapped poll votes",
                "tracts": N_TRACTS, "top_tracts": n_top, "top_cut_pct": round(float(cut), 1), "city_median_chow_pct": round(float(S5.chow_share.median()), 1),
                "sources": "resultsbyctuid2021_2023/2026.dta; social-geography census.parquet (renters, household income, car commuting, visible minority); archive modelling curated CSV (median age)",
                "built_by": "analysis/election_2026/03_neighbourhood_types_page_data.ipynb"},
       "top": [int(t) for t in S5.top],
       "variables": [{"key": v["key"], "label": v["label"], "unit": v["unit"], "domain": v["domain"], "tick": v["tick"],
                      "median_all": round(float(S5[v["key"]].median()), 1) if v["unit"] != "dollars" else int(round(S5[v["key"]].median())),
                      "median_top": round(float(S5.loc[S5.top, v["key"]].median()), 1) if v["unit"] != "dollars" else int(round(S5.loc[S5.top, v["key"]].median())),
                      "outside": int((S5[v["key"]].notna() & ~S5[v["key"]].between(*v["domain"])).sum()),
                      "values": clean(S5[v["key"]], v)} for v in STRIP_VARS]}
path = ROUTE_DATA / "chow_top_fifth_strips.json"
path.write_text(json.dumps(out, ensure_ascii=False, separators=(",", ":")) + "\n")
back = json.loads(path.read_text())
assert sum(back["top"]) == n_top and all(len(v["values"]) == N_TRACTS for v in back["variables"])
print(f"wrote {path.relative_to(REPO_ROOT)} ({path.stat().st_size:,} bytes)")

wrote src/routes/the-2026-mayoral-election-neighbourhood-types/data/chow_top_fifth_strips.json (17,778 bytes)
